In [0]:
# Célula 1 — Lendo a camada Bronze
BRONZE_PATH = "/Volumes/workspace/default/raw/bronze/"          # caminho da camada Bronze

df_silver = spark.read.format("delta").load(BRONZE_PATH)        # lê o Delta Lake da Bronze

print(f"Linhas: {df_silver.count()}")                           # conta as linhas
print(f"Colunas: {df_silver.columns}")                          # lista as colunas
df_silver.printSchema()                                         # mostra tipos de cada coluna

Linhas: 8469
Colunas: ['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Customer_Age', 'Customer_Gender', 'Product_Purchased', 'Date_of_Purchase', 'Ticket_Type', 'Ticket_Subject', 'Ticket_Description', 'Ticket_Status', 'Resolution', 'Ticket_Priority', 'Ticket_Channel', 'First_Response_Time', 'Time_to_Resolution', 'Customer_Satisfaction_Rating']
root
 |-- Ticket_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Customer_Email: string (nullable = true)
 |-- Customer_Age: string (nullable = true)
 |-- Customer_Gender: string (nullable = true)
 |-- Product_Purchased: string (nullable = true)
 |-- Date_of_Purchase: string (nullable = true)
 |-- Ticket_Type: string (nullable = true)
 |-- Ticket_Subject: string (nullable = true)
 |-- Ticket_Description: string (nullable = true)
 |-- Ticket_Status: string (nullable = true)
 |-- Resolution: string (nullable = true)
 |-- Ticket_Priority: string (nullable = true)
 |-- Ticket_Channel: string (nullable = true)
 |-- Firs

In [0]:
from pyspark.sql.functions import col, to_timestamp, to_date  # funções de conversão de tipos

df_silver = (df_silver
    .withColumn("Ticket_ID",                    col("Ticket_ID").cast("integer"))             # converte para número inteiro
    .withColumn("Customer_Age",                 col("Customer_Age").cast("integer"))           # converte para número inteiro
    .withColumn("Customer_Satisfaction_Rating", col("Customer_Satisfaction_Rating").cast("double"))  # converte para decimal
    .withColumn("Date_of_Purchase",             to_date("Date_of_Purchase", "yyyy-MM-dd"))    # converte para data
    .withColumn("First_Response_Time",          to_timestamp("First_Response_Time"))          # converte para data + hora
    .withColumn("Time_to_Resolution",           to_timestamp("Time_to_Resolution"))           # converte para data + hora
)

df_silver.printSchema()                                                                       # valida os novos tipos

root
 |-- Ticket_ID: integer (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Customer_Email: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Customer_Gender: string (nullable = true)
 |-- Product_Purchased: string (nullable = true)
 |-- Date_of_Purchase: date (nullable = true)
 |-- Ticket_Type: string (nullable = true)
 |-- Ticket_Subject: string (nullable = true)
 |-- Ticket_Description: string (nullable = true)
 |-- Ticket_Status: string (nullable = true)
 |-- Resolution: string (nullable = true)
 |-- Ticket_Priority: string (nullable = true)
 |-- Ticket_Channel: string (nullable = true)
 |-- First_Response_Time: timestamp (nullable = true)
 |-- Time_to_Resolution: timestamp (nullable = true)
 |-- Customer_Satisfaction_Rating: double (nullable = true)



In [0]:
from pyspark.sql.functions import when, lit  # funções condicionais

df_silver = (df_silver
    .withColumn("Resolution",                 when(col("Resolution").isNull(),                 lit("Sem resolução")).otherwise(col("Resolution")))                  # tickets ainda abertos
    .withColumn("Time_to_Resolution",         when(col("Time_to_Resolution").isNull(),         lit(None)).otherwise(col("Time_to_Resolution")))                     # mantém nulo — ticket aberto
    .withColumn("Customer_Satisfaction_Rating", when(col("Customer_Satisfaction_Rating").isNull(), lit(None)).otherwise(col("Customer_Satisfaction_Rating")))       # mantém nulo — sem avaliação
)

# Validando nulos restantes
from pyspark.sql.functions import count, isnan  

for c in df_silver.columns:
    nulos = df_silver.filter(col(c).isNull()).count()          # conta nulos por coluna
    if nulos > 0:
        print(f"{c}: {nulos} nulos")                           # exibe apenas colunas com nulos
        
print("✅ Tratamento de nulos concluído!")

First_Response_Time: 2819 nulos
Time_to_Resolution: 5700 nulos
Customer_Satisfaction_Rating: 5700 nulos
✅ Tratamento de nulos concluído!


In [0]:
from pyspark.sql.functions import current_timestamp  # data/hora atual

df_silver = df_silver.dropDuplicates(["Ticket_ID"])  # remove duplicatas pelo ID único

df_silver = df_silver.withColumn("_loaded_at", current_timestamp())  # registra quando foi processado

SILVER_PATH = "/Volumes/workspace/default/raw/silver/"

(df_silver.write
    .format("delta")                        # formato Delta Lake
    .mode("overwrite")                      # sobrescreve se já existir
    .option("overwriteSchema", "true")      # atualiza o schema se mudou
    .save(SILVER_PATH)
)

df_check = spark.read.format("delta").load(SILVER_PATH)
print(f"Linhas na Silver: {df_check.count()}")   # valida o total gravado
print("✅ Camada Silver gravada com sucesso!")

Linhas na Silver: 8469
✅ Camada Silver gravada com sucesso!


In [0]:
from pyspark.sql.functions import regexp_replace          # importa função de substituição por expressão regular

df_silver = df_silver.withColumn(
    "Ticket_Description",                                 # coluna que será substituída
    regexp_replace(
        "Ticket_Description",                             # coluna de origem
        r"\{product_purchased\}",                         # padrão a encontrar — chaves escapadas com \
        "[produto]"                                       # texto substituto
    )
)

# Validando — não deve aparecer mais nenhum {product_purchased}
remaining = df_silver.filter(
    df_silver["Ticket_Description"].contains("{product_purchased}")  # filtra linhas que ainda têm o placeholder
).count()                                                            # conta quantas sobraram

print(f"Placeholders restantes: {remaining}")                        # esperado: 0
print("✅ Ticket_Description limpa!")

Placeholders restantes: 0
✅ Ticket_Description limpa!


In [0]:
df_silver.show(10, truncate=50)                               # mostra 10 linhas, trunca texto em 50 caracteres

+---------+----------------+---------------------------+------------+---------------+-------------------------+----------------+---------------+------------------------+--------------------------------------------------+-------------------------+--------------------------------------------------+---------------+--------------+-------------------+-------------------+----------------------------+--------------------------+
|Ticket_ID|   Customer_Name|             Customer_Email|Customer_Age|Customer_Gender|        Product_Purchased|Date_of_Purchase|    Ticket_Type|          Ticket_Subject|                                Ticket_Description|            Ticket_Status|                                        Resolution|Ticket_Priority|Ticket_Channel|First_Response_Time| Time_to_Resolution|Customer_Satisfaction_Rating|                _loaded_at|
+---------+----------------+---------------------------+------------+---------------+-------------------------+----------------+---------------+------

In [0]:
# Salvando a Silver final no Delta Lake
(df_silver.write
    .format("delta")                          # formato Delta Lake
    .mode("overwrite")                        # sobrescreve a versão anterior
    .option("overwriteSchema", "true")        # atualiza o schema se mudou
    .save(SILVER_PATH)
)

# Validando o que foi salvo
df_check = spark.read.format("delta").load(SILVER_PATH)       # relê do disco
print(f"Linhas na Silver: {df_check.count()}")                # esperado: 8469
print(f"Colunas: {len(df_check.columns)}")                    # esperado: 18
print("✅ Camada Silver salva com sucesso!")

Linhas na Silver: 8469
Colunas: 18
✅ Camada Silver salva com sucesso!


### Testes de Qualidade — Data Quality Checks

Validações automáticas aplicadas após o ETL para garantir
a integridade dos dados antes de salvar na camada Silver.

In [0]:
import pyspark.sql.functions as F
import builtins

print("=" * 60)
print("DATA QUALITY CHECKS — CAMADA SILVER")
print("=" * 60)

erros = []

# Teste 1 — Volume
total = df_silver.count()
esperado = 8469
if total == esperado:
    print(f"✅ Teste 1 — Volume: {total:,} registros")
else:
    erros.append(f"🔴 Teste 1 — Volume incorreto: {total:,}")
    print(erros[-1])

# Teste 2 — Ticket_ID sem nulos
nulos_id = df_silver.filter(F.col("Ticket_ID").isNull()).count()
if nulos_id == 0:
    print(f"✅ Teste 2 — Ticket_ID sem nulos")
else:
    erros.append(f"🔴 Teste 2 — Ticket_ID com nulos: {nulos_id}")
    print(erros[-1])

# Teste 3 — Ticket_ID único
unicos = df_silver.select("Ticket_ID").distinct().count()
if unicos == total:
    print(f"✅ Teste 3 — Ticket_ID único: {unicos:,} IDs distintos")
else:
    erros.append(f"🔴 Teste 3 — Duplicatas: {total - unicos}")
    print(erros[-1])

# Teste 4 — Customer_Age
age_anomalias = df_silver.filter(
    (F.col("Customer_Age") < 18) | (F.col("Customer_Age") > 100)
).count()
if age_anomalias == 0:
    print(f"✅ Teste 4 — Customer_Age no range (18-100)")
else:
    erros.append(f"🔴 Teste 4 — Age anomalias: {age_anomalias}")
    print(erros[-1])

# Teste 5 — Satisfaction range
df_sat_check = df_silver.filter(F.col("Customer_Satisfaction_Rating").isNotNull())
sat_anomalias = df_sat_check.filter(
    (F.col("Customer_Satisfaction_Rating") < 1) |
    (F.col("Customer_Satisfaction_Rating") > 5)
).count()
if sat_anomalias == 0:
    print(f"✅ Teste 5 — Satisfaction no range (1-5)")
else:
    erros.append(f"🔴 Teste 5 — Satisfaction anomalias: {sat_anomalias}")
    print(erros[-1])

DATA QUALITY CHECKS — CAMADA SILVER
✅ Teste 1 — Volume: 8,469 registros
✅ Teste 2 — Ticket_ID sem nulos
✅ Teste 3 — Ticket_ID único: 8,469 IDs distintos
✅ Teste 4 — Customer_Age no range (18-100)
✅ Teste 5 — Satisfaction no range (1-5)


In [0]:
# Teste 6 — Placeholder
placeholder = df_silver.filter(
    F.col("Ticket_Description").contains("{product_purchased}")
).count()
if placeholder == 0:
    print(f"✅ Teste 6 — Placeholder removido")
else:
    erros.append(f"🔴 Teste 6 — Placeholder presente: {placeholder}")
    print(erros[-1])

# Teste 7 — Resolution sem nulos
nulos_res = df_silver.filter(F.col("Resolution").isNull()).count()
if nulos_res == 0:
    print(f"✅ Teste 7 — Resolution sem nulos")
else:
    erros.append(f"🔴 Teste 7 — Resolution com nulos: {nulos_res}")
    print(erros[-1])

# Teste 8 — Closed vs Satisfaction
closed_sem_sat = df_silver.filter(
    (F.col("Ticket_Status") == "Closed") &
    (F.col("Customer_Satisfaction_Rating").isNull())
).count()
if closed_sem_sat == 0:
    print(f"✅ Teste 8 — Tickets Closed têm Satisfaction Rating")
else:
    erros.append(f"🔴 Teste 8 — Closed sem Satisfaction: {closed_sem_sat}")
    print(erros[-1])

# Teste 9 — _loaded_at
nulos_loaded = df_silver.filter(F.col("_loaded_at").isNull()).count()
if nulos_loaded == 0:
    print(f"✅ Teste 9 — _loaded_at preenchido")
else:
    erros.append(f"🔴 Teste 9 — _loaded_at com nulos: {nulos_loaded}")
    print(erros[-1])

# Resultado final
print()
print("=" * 60)
if len(erros) == 0:
    print("🎉 TODOS OS TESTES PASSARAM — Silver aprovada para Gold!")
else:
    print(f"🔴 {len(erros)} TESTE(S) FALHARAM!")
    for e in erros:
        print(f"   {e}")
print("=" * 60)

✅ Teste 6 — Placeholder removido
✅ Teste 7 — Resolution sem nulos
✅ Teste 8 — Tickets Closed têm Satisfaction Rating
✅ Teste 9 — _loaded_at preenchido

🎉 TODOS OS TESTES PASSARAM — Silver aprovada para Gold!
